In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import time
from tqdm.auto import tqdm

In [ ]:

# Fast TQDM RandomizedSearchCV (Helper Class)

class FastTQDMSearchCV(RandomizedSearchCV):
    """RandomizedSearchCV with lightweight tqdm progress bar."""
    def fit(self, X, y=None, **fit_params):
        total_candidates = getattr(self, "n_iter", None)
        if total_candidates is None:
            if isinstance(self.param_distributions, list):
                total_candidates = sum(len(d) for d in self.param_distributions if isinstance(d, dict))
            else:
                total_candidates = 10
        pbar = tqdm(total=total_candidates, desc="Tuning Progress", leave=True)
        old_run_search = self._run_search
        def new_run_search(evaluate_candidates):
            def wrapped(candidate_params):
                results = evaluate_candidates(candidate_params)
                pbar.update(len(candidate_params))
                return results
            old_run_search(wrapped)
        self._run_search = new_run_search
        try:
            result = super().fit(X, y, **fit_params)
        finally:
            pbar.close()
        return result

In [ ]:

#  Load Original Data

print("Loading data...")
try:
    X_train_unscaled = pd.read_parquet('X_train.parquet')
    y_train = pd.read_parquet('y_train.parquet').values.ravel()
    X_test_unscaled = pd.read_parquet('X_test.parquet')
    y_test = pd.read_parquet('y_test.parquet').values.ravel()
except FileNotFoundError:
    print("FATAL ERROR: Could not find original X_train/X_test.parquet files.")
    raise

print(f"X_train shape:{X_train_unscaled.shape}")
print(f"y_train shape:{y_train.shape}")
print(f"X_test shape:{X_test_unscaled.shape}")
print(f"y_train shape:{y_test.shape}")

Loading data...
X_train shape:(1241213, 109)
y_train shape:(1241213,)
X_test shape:(310997, 109)
y_train shape:(310997,)


In [ ]:

# SHAP Feature Selection

TOP_20_FEATURES = [
    'HCO3_max_12hr',
    'Chloride_max_12hr',
    'Temp_mean_12hr',
    'HospAdmTime',
    'HCO3_mean_12hr',
    'EtCO2_mean_12hr',
    'WBC_mean_12hr',
    'Platelets_max_12hr',
    'DBP_max_12hr',
    'pH_max_12hr',
    'Temp_max_12hr',
    'Magnesium_max_12hr',
    'Phosphate_mean_12hr',
    'O2Sat_std_12hr',
    'Age',
    'EtCO2_max_12hr',
    'Fibrinogen_std_12hr',
    'SBP_max_12hr',
    'Phosphate_max_12hr',
    'PTT_mean_12hr'
]
# ----------------------------------------------------

print(f"Filtering data to Top {len(TOP_20_FEATURES)} features...")
X_train_lite = X_train_unscaled[TOP_20_FEATURES]
# We also filter the test set now
X_test_lite = X_test_unscaled[TOP_20_FEATURES]

Filtering data to Top 20 features...


In [ ]:
# Apply SMOTE to "Lite" Data

# We'll aim for a 1:10 ratio (1 sepsis case for every 10 healthy)
smote = SMOTE(sampling_strategy=0.1, random_state=42)
print("\nStarting SMOTE... (This may take a few minutes)")
start_time = time.time()

X_train_smote, y_train_smote = smote.fit_resample(X_train_lite, y_train)
print(f"SMOTE complete in {(time.time() - start_time):.1f}s")
print(f"Original shape: {X_train_lite.shape} -> New shape: {X_train_smote.shape}")

try:
    total_smote_samples = len(y_train_smote) 

    # Count of positive and negative cases
    positive_smote_count = np.sum(y_train_smote == 1)
    negative_smote_count = np.sum(y_train_smote == 0)

    # Calculate the new ratio
    ratio_neg_to_pos = negative_smote_count / positive_smote_count if positive_smote_count > 0 else np.nan

    print(f"--- SMOTE Balance Check ---")
    print(f"Total samples (rows): {total_smote_samples}")
    print(f"Positive cases (1): {positive_smote_count}")
    print(f"Negative cases (0): {negative_smote_count}")
    print(f"New Ratio (Negatives per Positive): {ratio_neg_to_pos:.2f} to 1")
    print(f"Target Ratio: 10.00 to 1 (from sampling_strategy=0.1)")

except NameError:
    print("ERROR: 'y_train_smote' is not defined. Please run the SMOTE part of the script first.")


Starting SMOTE... (This may take a few minutes)
SMOTE complete in 1.4s
Original shape: (1241213, 20) -> New shape: (1335144, 20)
--- SMOTE Balance Check ---
Total samples (rows): 1335144
Positive cases (1): 121376
Negative cases (0): 1213768
New Ratio (Negatives per Positive): 10.00 to 1
Target Ratio: 10.00 to 1 (from sampling_strategy=0.1)


In [ ]:
# Scale the NEW SMOTE Data

print("Scaling new SMOTE data...")
# We create a NEW scaler and fit it ONLY on the new, balanced data
scaler_smote = StandardScaler()
X_train_scaled_smote = scaler_smote.fit_transform(X_train_smote)

# We use this NEW scaler to transform our original test set
X_test_scaled = scaler_smote.transform(X_test_lite)

Scaling new SMOTE data...


In [ ]:
# Re-Tune XGBoost on "Lite+SMOTE" Data

print(f"\n--- Tuning XGBoost on {X_train_scaled_smote.shape[0]} 'Lite+SMOTE' samples ---")
start_time = time.time()
ratio = float(np.sum(y_train_smote == 0)) / np.sum(y_train_smote == 1) # Should be ~10.0
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Use the same grid as before
xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_search = FastTQDMSearchCV(
    XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=42),
    xgb_param_grid,
    n_iter=10, # 10 iterations is fine
    scoring='roc_auc',
    cv=cv,
    n_jobs=1, # Keep search single-threaded
    random_state=42
)

xgb_search.fit(X_train_scaled_smote, y_train_smote)
print(f"New tuning time: {(time.time() - start_time):.1f}s")
best_xgb_new_params = xgb_search.best_params_
print(f"Best NEW Params: {best_xgb_new_params}")
print(f"Best NEW CV AUROC: {xgb_search.best_score_:.4f}")


--- Tuning XGBoost on 1335144 'Lite+SMOTE' samples ---


Tuning Progress:   0%|          | 0/10 [00:00<?, ?it/s]

New tuning time: 84.7s
Best NEW Params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
Best NEW CV AUROC: 0.7403


In [ ]:
# Train Final Model with NEW Params

print("\n--- Training final model on FULL 'Lite+SMOTE' dataset ---")
final_xgb_new = XGBClassifier(
    **best_xgb_new_params,
    scale_pos_weight=ratio,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1 
)
final_xgb_new.fit(X_train_scaled_smote, y_train_smote, verbose=True)
print("✅ New model training complete.")


--- Training final model on FULL 'Lite+SMOTE' dataset ---


c:\Users\shirs\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [02:33:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ New model training complete.


In [ ]:
# Final Evaluation

print("\n--- Final Test Set Evaluation (SHAP+SMOTE Model) ---")
xgb_probs_new = final_xgb_new.predict_proba(X_test_scaled)[:, 1]
xgb_auroc_new = roc_auc_score(y_test, xgb_probs_new)
xgb_auprc_new = average_precision_score(y_test, xgb_probs_new)

print(f"XGBoost (Lite+SMOTE) Test AUROC: {xgb_auroc_new:.4f}")
print(f"XGBoost (Lite+SMOTE) Test AUPRC: {xgb_auprc_new:.4f}")

print("\n--- Comparison ---")
try:
    print(f"Original AUPRC (Full Data): 0.0094") # 'xgb_final_auprc' is from your previous cell
except NameError:
    print("Could not find 'xgb_final_auprc' in memory. Please check your previous run's AUPRC.")

print(f"New AUPRC (Lite+SMOTE):   {xgb_auprc_new:.4f}")


--- Final Test Set Evaluation (SHAP+SMOTE Model) ---
XGBoost (Lite+SMOTE) Test AUROC: 0.7412
XGBoost (Lite+SMOTE) Test AUPRC: 0.0093

--- Comparison ---
Original AUPRC (Full Data): 0.0094
New AUPRC (Lite+SMOTE):   0.0093
